In [1]:
import pandas as pd
from catboost import CatBoostClassifier

#### Load the train data and drop irrelevant features


In [2]:
df = pd.read_csv('train.csv',sep=',')
y = df["Survived"]
df = df.drop(columns=["PassengerId","Name","Survived","Cabin","Ticket"],axis=1)

- There are some missing values in the Age column, so I decided to fill them with the mean age of each Pclass group.


In [ ]:
print(df.isna().sum())
df_invalid_age=df[df['Age'].isna()] ##create a df for the data with NaN age
print(df_invalid_age["Pclass"].value_counts())##checking which Pclass has the most NaN age

mean_ages = df.groupby("Pclass")["Age"].mean()

## Filling the Nan ages
df["Age"] = df.apply(
    lambda row: mean_ages[row["Pclass"]] if pd.isna(row["Age"]) else row["Age"], axis=1
)


There are only 2 missing data points, so I decided to examine each individually. and then fill based on the mode of their Pclass

In [ ]:
print(df[df["Embarked"].isna()]) #it seems that both missing data are Pclass = 1
print(df[df["Pclass"]==1]["Embarked"].value_counts()) ## checking the most common embarked value for first class
df["Embarked"] = df["Embarked"].fillna(df[df["Pclass"]==1]["Embarked"].mode()[0]) # fill with the Pclass==1 mode
X_train = df

### Load the test dataset and adjust it to match the shape of the train dataset.

In [5]:
df_test = pd.read_csv('test.csv',sep=',')
passenger_id = df_test["PassengerId"]
df_test = df_test.drop(['Name','Cabin','PassengerId',"Ticket"],axis=1)

### Applying the same changes


In [ ]:
print(df_test.isna().sum())
df_test_invalid_age=df_test[df_test['Age'].isna()]
print(df_test_invalid_age["Pclass"].value_counts())

mean_ages = df_test.groupby("Pclass")["Age"].mean()

df_test["Age"] = df_test.apply(
    lambda row: mean_ages[row["Pclass"]] if pd.isna(row["Age"]) else row["Age"], axis=1
)

print(df_test[df_test["Embarked"].isna()])
print(df_test[df_test["Pclass"]==1]["Embarked"].value_counts())
df_test["Embarked"] = df_test["Embarked"].fillna(df_test[df_test["Pclass"]==1]["Embarked"].mode()[0])
X_test = df_test

Choose the Model, train and predict

In [ ]:
model = CatBoostClassifier(iterations= 1000,learning_rate=0.1,cat_features=["Embarked","Sex"],task_type='GPU',loss_function="Logloss")
model.fit(X_train, y, plot=True)
y_pred = model.predict(X_test)

submission = pd.DataFrame({
   'PassengerId': passenger_id,  
  'Survived': y_pred                 
})
submission.to_csv('submission.csv',index=False)